In [1]:
import pandas as pd
from pathlib import Path

data_path = Path("../data/raw/chennai/bus")

routes = pd.read_csv(data_path / "routes.txt", dtype=str)
trips = pd.read_csv(data_path / "trips.txt", dtype=str)
stops = pd.read_csv(data_path / "stops.txt", dtype=str)
stop_times = pd.read_csv(data_path / "stop_times.txt", dtype=str)

print("Routes:", routes.shape)
print("Trips:", trips.shape)
print("Stops:", stops.shape)
print("Stop times:", stop_times.shape)
print("\nStop times columns:", stop_times.columns.tolist())

Routes: (4611, 5)
Trips: (47047, 4)
Stops: (5580, 4)
Stop times: (1360443, 5)

Stop times columns: ['trip_id', 'arrival_time', 'departure_time', 'stop_id', 'stop_sequence']


In [2]:
# Count distinct trips per route
trip_counts = trips.groupby("route_id")["trip_id"].nunique().reset_index(name="total_trips")

# Count distinct stops per route
trip_stops = trips[["route_id", "trip_id"]].merge(
    stop_times[["trip_id", "stop_id"]],
    on="trip_id",
    how="left"
)

stop_counts = trip_stops.groupby("route_id")["stop_id"].nunique().reset_index(name="total_stops")

# Combine route details and features
route_features = routes.merge(trip_counts, on="route_id", how="left")
route_features = route_features.merge(stop_counts, on="route_id", how="left")

print(route_features.head())

  route_id route_short_name  \
0    18751              29A   
1    18750              12G   
2    23728             11 R   
3    18753            22 CT   
4    20494           15 CT8   

                                     route_long_name route_type agency_id  \
0                        Anna Square TO Perambur B.T          3        69   
1  Sathya Nagar TO Lady Willingdon College Of Edu...          3        69   
2                      Royapuram TO Thiyagarayanagar          3        69   
3                          Anna Square TO Ayanavaram          3        69   
4                 Anna Nagar West TO M.G.R.Koyambedu          3        69   

   total_trips  total_stops  
0        106.0         29.0  
1          1.0         29.0  
2         40.0         22.0  
3          1.0         25.0  
4         16.0          4.0  


In [3]:
def gtfs_hour(t):
    try:
        h, m, s = map(int, t.split(":"))
        return h + m / 60 + s / 3600
    except:
        return None

stop_times["arrival_hour"] = stop_times["arrival_time"].apply(gtfs_hour)
stop_times["departure_hour"] = stop_times["departure_time"].apply(gtfs_hour)

def time_period(h):
    if pd.isna(h):
        return "Unknown"
    if 5 <= h < 12:
        return "Morning"
    if 12 <= h < 17:
        return "Afternoon"
    if 17 <= h < 21:
        return "Evening"
    return "Night"

stop_times["arrival_period"] = stop_times["arrival_hour"].apply(time_period)
stop_times["departure_period"] = stop_times["departure_hour"].apply(time_period)

first_departures = stop_times.sort_values("stop_sequence").drop_duplicates("trip_id")

trip_hours = first_departures.merge(
    trips[["trip_id", "route_id"]],
    on="trip_id",
    how="left"
)

hourly_trips = (
    trip_hours.groupby(["route_id", "departure_hour"])["trip_id"]
    .nunique()
    .reset_index(name="trip_count")
)

print(stop_times[["arrival_hour", "departure_hour", "departure_period"]].head())
print("\nHourly trips:")
print(hourly_trips.head())

   arrival_hour  departure_hour departure_period
0      6.333333        6.333333          Morning
1      6.366667        6.366667          Morning
2      6.383333        6.383333          Morning
3      6.400000        6.400000          Morning
4      6.433333        6.433333          Morning

Hourly trips:
  route_id  departure_hour  trip_count
0    10003       20.333333           1
1    10083        4.916667           1
2    10083        5.333333           1
3    10083        6.916667           1
4    10084        6.083333           1


In [4]:
output_path = Path("../data/processed/chennai")
output_path.mkdir(parents=True, exist_ok=True)

route_features.to_csv(output_path / "route_profile.csv", index=False)
hourly_trips.to_csv(output_path / "hourly_trip_counts.csv", index=False)
stop_times.to_csv(output_path / "time_features.csv", index=False)

print("Chennai CSV files saved successfully!")

print("\nMissing values:\n", route_features.isnull().sum())
print("\nDuplicate route IDs:", route_features["route_id"].duplicated().sum())
print("Duplicate trip IDs:", trips["trip_id"].duplicated().sum())
print("Invalid arrival hours:", stop_times["arrival_hour"].isna().sum())
print("Invalid departure hours:", stop_times["departure_hour"].isna().sum())

Chennai CSV files saved successfully!

Missing values:
 route_id              0
route_short_name      0
route_long_name       0
route_type            0
agency_id             0
total_trips         677
total_stops         677
dtype: int64

Duplicate route IDs: 0
Duplicate trip IDs: 0
Invalid arrival hours: 0
Invalid departure hours: 0


In [5]:
def gtfs_minute(t):
    try:
        h, m, s = map(int, t.split(":"))
        return m
    except:
        return None

stop_times["arrival_minute"] = stop_times["arrival_time"].apply(gtfs_minute)
stop_times["departure_minute"] = stop_times["departure_time"].apply(gtfs_minute)

stop_times["stop_sequence"] = pd.to_numeric(
    stop_times["stop_sequence"], errors="coerce"
)

first_departures = (
    stop_times.sort_values("stop_sequence")
    .drop_duplicates("trip_id")
)

first_departures["departure_service_hour"] = (
    first_departures["departure_hour"].apply(
        lambda h: int(h) if pd.notna(h) else None
    )
)

trip_hours = first_departures.merge(
    trips[["trip_id", "route_id"]],
    on="trip_id",
    how="left"
)

hourly_trips = (
    trip_hours.groupby(["route_id", "departure_service_hour"])["trip_id"]
    .nunique()
    .reset_index(name="trip_count")
)

output_path = Path("../data/processed/chennai")
hourly_trips.to_csv(output_path / "hourly_trip_counts.csv", index=False)
stop_times.to_csv(output_path / "time_features.csv", index=False)

print("Chennai updated successfully!")

Chennai updated successfully!
